## HSP Image Generator

#### Hierarchical Spatial Pooler trained on celebA dataset

In [ ]:
import sys, subprocess
sys.path.insert(0, subprocess.check_output(['git', 'rev-parse', '--show-toplevel']).decode().strip())

In [ ]:
import os
from typing import Sequence

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import torch
from tqdm import tqdm
from matplotlib import pyplot as plt

tqdm.pandas()
%matplotlib inline

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='htm')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
seed = 42
torch.manual_seed(seed)

from dataloaders import get_replicated_dataloader
from models import HTMModel, CNNModel, CNNTrainer
from utils import generate_config

### Dataloader for celebA

Object of dataloader with specific adjustments:

* Load celebA with Dataset - Grayscale (optional)
* Transform - default transform reshapes images to 64 * 64 and convert to binary.

In [ ]:

mixed_samples_indices = list(range(0,240, 1))
num_samples = 9600

# dataset_name='cifar10'
dataset_name='celeba'

dataloader = get_replicated_dataloader(
  dataset_name='celeba',
  sample_indices=mixed_samples_indices,
  total_size=9600,
  download=True
)

dataloader.dataset.return_idx = True


### Preprocessing data:

Using CelebADataloader and PreProcessor:
* Create adjacent 80% masking tensor for each sample
* Split image
* Apply SP on each quarter
* Merge parts for next layer


In [ ]:
save_path_for_curr_run = 'e11_vertical_htm40_train_cnn'

# Generate config with all computed dimensions
config = generate_config(
    image_size=64,
    base_enc=4,
    split_factor=4,
    channels=1,
    cnn={"inner_dim": 64, "learning_rate": 0.001},
    training={"batch_size": 64, "patience": 10}
)

# Additional runtime settings (not part of config)
runtime_config = {
    'epochs': [0, 100],
    'device': 'mps' if torch.mps.is_available() else 'cpu',
    'save_path': save_path_for_curr_run,
    'mask_ratio': 0.15
}

In [ ]:
config.training

## Save/Load Trained HTM Model

Functions to save and load the trained HTM preprocessor. This preserves all learned spatial pooler states so you can reuse the trained HTM without retraining.

In [ ]:
import pickle


def save_htm_model(htm_model, save_path, model_name="trained_htm"):
    """
    Save the trained HTM preprocessor with all learned spatial pooler states.

    Args:
        htm_model: The HTMModel instance with trained spatial poolers
        save_path: Directory to save the model (e.g., 'results/e7_celeba_10_large')
        model_name: Name for the saved model file
    """
    os.makedirs(save_path, exist_ok=True)

    model_path = os.path.join(save_path, f"{model_name}.pkl")

    # Save the entire HTM preprocessor (includes learned spatial poolers)
    with open(model_path, 'wb') as f:
        pickle.dump(htm_model, f)

    print(f"HTM model saved to {model_path}")
    return model_path


def load_htm_model(model_path):
    """
    Load a previously trained HTM preprocessor.

    Args:
        model_path: Full path to the saved model file

    Returns:
        HTMModel instance with learned state restored
    """
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")

    with open(model_path, 'rb') as f:
        htm_model = pickle.load(f)

    print(f"HTM model loaded from {model_path}")
    return htm_model

In [ ]:
import os

saved_htm_path = 'e11_celeba_vertical'

# Check if trained HTM model exists
htm_model_path = f"../results/{saved_htm_path}/trained_htm_epoch_s128_40.pkl"

if os.path.exists(htm_model_path):
    print(f"Loading existing HTM model from {htm_model_path}")
    htm_model = load_htm_model(htm_model_path)
else:
    print("No existing HTM model found. Initializing new HTM preprocessor...")
    htm_model = HTMModel(
        config=config,
        mask_ratio=runtime_config['mask_ratio'],
        dataset_size=num_samples,
        device=runtime_config['device'],
        use_cache=True
    )
    print("HTM will be trained during the training process")

### Generative Model for Images with HTM-CNN pipeline:


In [ ]:
# Initialize CNN model from config
model = CNNModel(config=config)

# Initialize CNN trainer with HTM preprocessor and config
trainer = CNNTrainer(
    model=model,
    htm_model=htm_model,
    config=config
)


## cnn 0.15 mse / 0.85 ssim

In [ ]:
htm_model.learning_enabled = False

# Initialize CNN model from config
model = CNNModel(config=config)

# Initialize CNN trainer with HTM preprocessor and config
trainer = CNNTrainer(
    model=model,
    htm_model=htm_model,
    config=config
)

trainer.train(dataloader=dataloader, epochs=[0,250], save_path=runtime_config['save_path'] + '_wo')

In [ ]:
htm_model.learning_enabled = False

from htm.bindings.sdr import SDR
def to_sdr(t):
    """Convert tensor to SDR if needed."""
    if isinstance(t, torch.Tensor):
        arr = t.cpu().numpy()
        sdr = SDR(arr.shape)
        sdr.dense = arr
        return sdr
    return t

from utils import HTMAnalyzer

# Get n outputs from your model
analyzer = HTMAnalyzer()

batch_tensors = model.generate_sdr(10, htm_model, rand_mask=True)
batch = [to_sdr(t) for t in batch_tensors]

metrics = analyzer.track_sdr_sequence(batch, name="gen_analysis")
analyzer.plot_overlap_matrix(batch)

# Generate and plot sample images using the trained model
generated_images = model.generate_from_SDR(batch_tensors)
analyzer.plot_sdrs_grid(batch, ncols=5)
model.plot_images(generated_images)

htm_model.learning_enabled = True


# generate:

In [ ]:
# Generate and plot sample images using the trained model
generated_images = model.generate(num_samples=50, htm_model=htm_model)
model.plot_images(generated_images)

In [ ]:
# Generate and plot sample images using the trained model
generated_images = model.generate(num_samples=10, htm_model=htm_model)
model.plot_images(generated_images)